# 2. Collect demonstrations with SO-101

The **leader** is moved by the operator; the powered **follower** copies it while LeRobot records camera observations, joint states, actions and a task instruction. This notebook assumes two assembled SO-101 arms and two USB cameras: a top view and a wrist view.

**Safety:** Secure both arms; remove obstacles and people from the follower's reach. Keep a person at the power switch throughout. Start with small, slow motions and stop/cut power if motion is unexpected. Never force joints past their mechanical limits. Use an interactive terminal for all commands below, with the environment from notebook 1 activated.

## Discover ports and cameras

Connect the arms to USB and their appropriate power supplies. Run the port finder **once for each arm**, unplugging only the requested arm when prompted; reconnect it afterward. Write down which port belongs to which arm. On Linux, configure serial device permissions as described in the [SO-101 guide](https://huggingface.co/docs/lerobot/en/so101); on macOS, grant camera access if prompted.

```bash
lerobot-find-port
lerobot-find-port
lerobot-find-cameras opencv
```

Inspect the camera discovery output and captured images to identify the top and wrist camera indices. Indices and USB port names can change after reconnecting. Freshly assembled or unconfigured motors must first be set up one at a time following the [motor setup instructions](https://huggingface.co/docs/lerobot/en/so101#configure-the-motors); do not run motor setup on an already configured arm casually.

## Set your session values

Edit the example ports and camera indices below before pasting into the **same terminal**. Use stable, distinct IDs; calibration is stored per ID. These values also define the dataset and must remain consistent in the later notebooks. The `workshop/` namespace is just a local identifier here, not a Hub upload.

```bash
export FOLLOWER_PORT=/dev/ttyACM0  # replace with your follower port
export LEADER_PORT=/dev/ttyACM1    # replace with your leader port
export FOLLOWER_ID=workshop_follower
export LEADER_ID=workshop_leader
export DATASET_ID=workshop/so101-pick-place
export DATASET_ROOT="$HOME/lerobot-workshop-data/so101-pick-place"
export TASK='Pick up the cube and place it in the bin.'
export CAMERAS='{top: {type: opencv, index_or_path: 0, width: 640, height: 480, fps: 30}, wrist: {type: opencv, index_or_path: 1, width: 640, height: 480, fps: 30}}'
```

Replace camera indices `0` and `1` with those you discovered and confirm both cameras support 640×480 at 30 fps. Keep the names, viewpoints, resolution and task consistent during training and deployment.

Only if you are configuring new motors, follow the linked assembly guide's one-motor-at-a-time wiring and prompts before assembling the arms. The corresponding setup commands are:

```bash
lerobot-setup-motors --robot.type=so101_follower --robot.port="$FOLLOWER_PORT"
lerobot-setup-motors --teleop.type=so101_leader --teleop.port="$LEADER_PORT"
```

## Calibrate each arm

Run these separately, following the prompts for each arm. First place all joints approximately at the middle of their safe ranges, then press Enter. Move each joint, including the gripper, gently through its usable range without hitting stops, then finish when prompted. Watch the [calibration demonstration](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/lerobot/calibrate_so101_2.mp4). If offered an existing calibration, select recalibration only if needed. Keep the same IDs in all subsequent commands.

```bash
lerobot-calibrate --robot.type=so101_follower --robot.port="$FOLLOWER_PORT" --robot.id="$FOLLOWER_ID"
lerobot-calibrate --teleop.type=so101_leader --teleop.port="$LEADER_PORT" --teleop.id="$LEADER_ID"
```

## Test teleoperation

With a clear workspace and power cutoff ready, execute the following. Move the leader slowly and verify that the follower tracks correctly. Stop with Ctrl+C before recording. If joints behave unexpectedly, stop and check motor configuration, ports and calibration.

```bash
lerobot-teleoperate \
  --robot.type=so101_follower --robot.port="$FOLLOWER_PORT" --robot.id="$FOLLOWER_ID" \
  --robot.cameras="$CAMERAS" \
  --teleop.type=so101_leader --teleop.port="$LEADER_PORT" --teleop.id="$LEADER_ID"
```

## Record demonstrations

Start with a short pilot, inspect it, and then collect around 50 good demonstrations with variation in starting cube position. The example records 50 episodes of up to 60 seconds with 30 seconds to reset the scene. The fixed path must not already contain a dataset when starting a new recording. Record locally only; no Hub login is required.

```bash
lerobot-record \
  --robot.type=so101_follower --robot.port="$FOLLOWER_PORT" --robot.id="$FOLLOWER_ID" \
  --robot.cameras="$CAMERAS" \
  --teleop.type=so101_leader --teleop.port="$LEADER_PORT" --teleop.id="$LEADER_ID" \
  --dataset.repo_id="$DATASET_ID" --dataset.root="$DATASET_ROOT" \
  --dataset.no_stamp=true --dataset.push_to_hub=false \
  --dataset.single_task="$TASK" --dataset.fps=30 \
  --dataset.num_episodes=50 --dataset.episode_time_s=60 --dataset.reset_time_s=30
```

Keep the terminal focused: Right / `n` advances, Left / `r` discards and re-records the current episode, and Escape / `q` ends the session. Allow recording to finish saving. To add episodes, rerun the same command with `--resume=true` and set `--dataset.num_episodes` to the **additional** episode count. Do not remove the dataset to restart unless you have deliberately backed it up.

In [ ]:
from pathlib import Path

dataset_root = Path.home() / 'lerobot-workshop-data' / 'so101-pick-place'
print('Default dataset location:', dataset_root)
print('Dataset metadata exists:', (dataset_root / 'meta' / 'info.json').is_file())

If you changed `DATASET_ROOT`, update the check above. Inspect recorded videos and metadata for both camera views, task consistency and successful demonstrations before training. Data remains on the collection machine: copy the **entire dataset directory**, including metadata and videos, to the training machine at the same path, or change `DATASET_ROOT` in notebook 3. See the [LeRobot recording guide](https://huggingface.co/docs/lerobot/en/il_robots) for visualization and troubleshooting. Next: [training](03_training.ipynb).